# Cert Prep — 05: Structured Streaming

**Exam weight: ~13%**

Topics covered:
- Stream-as-table mental model
- readStream sources (file, socket, rate)
- Output modes: append, complete, update
- Triggers: default, processingTime, once, availableNow
- checkpointLocation — fault tolerance, exactly-once
- Watermarking — late data handling
- writeStream sinks


In [ ]:
import os, time, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = 'spark://spark-master:7077'

spark = (
    SparkSession.builder
    .appName('cert-prep')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.sql.warehouse.dir', '/workspace/data/training_warehouse')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

# Sample dataset used throughout
data = [
    (1, "Alice",  "Engineering", 95000.0, "2021-03-15", 4),
    (2, "Bob",    "Marketing",   72000.0, "2020-07-01", 3),
    (3, "Carol",  "Engineering", 105000.0,"2019-11-20", 5),
    (4, "Dave",   "Marketing",   68000.0, "2022-01-10", 2),
    (5, "Eve",    "Engineering", 88000.0, "2021-09-05", 4),
    (6, "Frank",  "HR",          61000.0, "2020-04-18", 3),
    (7, "Grace",  "HR",          None,    "2023-02-28", 1),
    (8, "Hank",   "Engineering", 112000.0,"2018-06-12", 5),
    (9, "Iris",   "Marketing",   None,    "2022-11-03", 2),
    (10,"Jack",   "HR",          59000.0, "2023-08-01", 1),
]
schema = StructType([
    StructField("id",         IntegerType(),  False),
    StructField("name",       StringType(),   True),
    StructField("dept",       StringType(),   True),
    StructField("salary",     DoubleType(),   True),
    StructField("hire_date",  StringType(),   True),
    StructField("perf_score", IntegerType(),  True),
])
df = spark.createDataFrame(data, schema)
df.cache().count()
print("Dataset ready")
df.show()

In [ ]:
# ── Output Modes ─────────────────────────────────────────────────────────────
print("=== Output Modes ===")
print()
print("append  — only NEW rows added since last trigger")
print("          use for: stateless queries, event logs, no aggregation")
print("          CANNOT use with aggregations (unless watermark defined)")
print()
print("complete — entire result table re-written every trigger")
print("           use for: aggregations without watermark")
print("           WARNING: grows unbounded for streaming sources")
print()
print("update  — only CHANGED rows since last trigger")
print("          use for: aggregations, key-value updates")
print("          most efficient — only writes deltas")
print()
print("Rules:")
print("  No aggregation   → append or update")
print("  Aggregation      → complete or update")
print("  Aggregation+wm   → append, complete, or update")

In [ ]:
# ── Triggers ─────────────────────────────────────────────────────────────────
print("=== Triggers ===")
print()
print("Trigger.Once()           — process all available data, then STOP")
print("                           batch compatibility for streaming backfill")
print()
print("Trigger.AvailableNow()   — like Once() but multi-batch, more efficient")
print("                           (Spark 3.3+)")
print()
print("Trigger.ProcessingTime('5 seconds')  — micro-batch every 5 sec")
print("                                       most common in production")
print()
print("Trigger.Continuous('1 second')       — experimental continuous mode")
print("                                       millisecond latency, limited ops")
print()
print("Default (no trigger specified)       — process as fast as possible")
print("                                       new batch starts immediately")
print()

# Rate source — generates rows at a fixed rate, useful for testing
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load())
print("Rate source schema:")
rate_stream.printSchema()

In [ ]:
# ── checkpointLocation — fault tolerance ─────────────────────────────────────
print("=== Checkpoint Location ===")
print()
print("checkpointLocation stores:")
print("  - Offsets: which source data has been read")
print("  - State:   aggregation state (for stateful queries)")
print("  - Commits: which micro-batches have been completed")
print()
print("Without checkpoint: stream restarts from scratch after failure")
print("With checkpoint:    stream resumes exactly where it left off")
print("                    → exactly-once processing guarantee")
print()
print("Configuration:")
print("  .writeStream")
print("    .option('checkpointLocation', '/path/to/checkpoint')")
print("    .format('delta')")
print("    .start()")
print()
print("IMPORTANT:")
print("  - checkpointLocation is set on writeStream (NOT readStream)")
print("  - Each streaming query needs its OWN checkpoint directory")
print("  - Changing query schema requires a new checkpoint location")

In [ ]:
# ── Watermarking — late data ─────────────────────────────────────────────────
print("=== Watermarking ===")
print()
print("Problem: streaming data arrives late (network delays, retries)")
print("Watermark: threshold of how late data is allowed to be")
print()
print("df.withWatermark('event_time', '10 minutes')")
print("  → Spark waits up to 10 min after max event time seen")
print("  → Data older than (max_event_time - 10min) is DROPPED")
print("  → Allows engine to clean up old state and output results")
print()
print("Without watermark:")
print("  - aggregations accumulate state forever (memory leak)")
print("  - append output mode NOT allowed for aggregations")
print()
print("With watermark:")
print("  - append mode allowed: results output after watermark passes")
print("  - update mode: results output every trigger, may be revised")
print("  - complete mode: full table every trigger (ignores watermark)")
print()

# Demonstrate watermark on rate source
from pyspark.sql.functions import window as W
wm_df = (rate_stream
    .withWatermark("timestamp", "30 seconds")
    .groupBy(W("timestamp", "10 seconds"))
    .count())
print("Streaming query plan with watermark:")
wm_df.explain()